<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/21_feature_importance_all_instruments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys, os
from pathlib import Path

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')
else:
    def _find_project_root():
        current = Path.cwd()
        for _ in range(10):
            if (current / "src").is_dir():
                return current
            current = current.parent
        return Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
    PROJ_ROOT = str(_find_project_root())

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

print("Project root:", PROJ_ROOT)
print("src exists:", os.path.isdir(os.path.join(PROJ_ROOT, 'src')))

In [ ]:
from src.experiments.helpers import TICKERS
print(TICKERS)

# 21 — Feature Importance on All Instruments

**Supervisor's question:** feature importance was only analysed on SPY. Are the same
indicators and pattern features important on other stocks, and does a feature set chosen
on SPY work as well as one chosen on many stocks?

**Method:** (1) rank features per instrument with a Random Forest, (2) compare the rankings
across instruments, (3) test whether SPY-derived or pool-derived feature sets generalise
better to instruments held out from training, (4) repeat the ranking on the bar-by-bar
dataset, including the recency features.

**Caveat to keep in mind:** each instrument has only ~130 to 160 events, so single-instrument
rankings are noisy. That is why the pooled ranking matters.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.experiments.helpers import (TICKERS, build_event_data, build_bar_data,
                                     align_columns, split_per_ticker)

event_data = {}
for t in TICKERS:
    event_data[t] = build_event_data(t)
    print(f"{t}: {event_data[t]['features'].shape[0]} events")

event_cols = align_columns(event_data)
print(f"\nCommon feature columns: {len(event_cols)}")